In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

root = Path.cwd()
for _ in range(6):
    if (root / 'data' / 'SampleSuperstore.csv').exists():
        break
    root = root.parent

print(" Performing Customer Analytics...")

# Load data
df = pd.read_csv(root / 'data' / 'SampleSuperstore.csv', encoding='latin-1')

# Display basic info
print("Dataset Overview:")
print(f"Shape: {df.shape}")
print("First 5 rows:")
display(df.head())

print("Basic Information:")
df.info()

# 1. Customer Analytics: RFM Analysis
print("Performing RFM Analysis...")
df['Order Date'] = pd.to_datetime(df['Order Date'])
max_date = df['Order Date'].max()

rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (max_date - x.max()).days,  # Recency
    'Order ID': 'count',                                # Frequency
    'Sales': 'sum'                                      # Monetary
}).rename(columns={'Order Date': 'Recency', 'Order ID': 'Frequency', 'Sales': 'Monetary'})

print("RFM Analysis Results:")
display(rfm.head())

# 2. Trend Analysis (Sales over time)
print("Performing Sales Trend Analysis...")
df['YearMonth'] = df['Order Date'].dt.to_period('M')
sales_trends = df.groupby('YearMonth').agg({'Sales': 'sum'}).reset_index()
sales_trends['YearMonth'] = sales_trends['YearMonth'].astype(str)

# Plot sales trends
plt.figure(figsize=(12, 6))
plt.plot(sales_trends['YearMonth'], sales_trends['Sales'], marker='o')
plt.title('Sales Trend Over Time')
plt.xlabel('Month')
plt.ylabel('Sales')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

# 3. Marketing Analytics: Analyze by Segment/Category
print("Performing Marketing Analytics...")
category_analysis = df.groupby('Category').agg({'Sales': 'sum', 'Profit': 'sum', 'Quantity': 'sum'}).reset_index()

print("Category Performance:")
display(category_analysis)

# Visualize category performance
fig = px.bar(category_analysis, x='Category', y='Sales', title='Sales by Category')
fig.show()

# 4. Save insights to Excel
print("Saving insights to Excel...")
report_dir = root / 'report'
report_dir.mkdir(exist_ok=True)
with pd.ExcelWriter(report_dir / 'customer_analytics_report.xlsx', engine='openpyxl') as writer:
    rfm.to_excel(writer, sheet_name='RFM Analysis')
    sales_trends.to_excel(writer, sheet_name='Sales Trends')
    category_analysis.to_excel(writer, sheet_name='Category Performance')

print(" Customer Analytics Complete! Excel file saved to report/customer_analytics_report.xlsx.")
